# 30C — Source-balance v2 replacement dense review

**CPU → Run all.** This evidence-only pass creates 2-FPS sheets for the three replacement sources advanced by 30B (Alex Crockford sit-up, Nona Bayat lat-pulldown, and SilverSneakers lat-pulldown) and retries the unavailable Max Euceda row locator with additional bounded yt-dlp fallbacks. It exports no raw video, landmarks, labels, or training data. Every row remains `training_allowed=false`.

In [ ]:
%pip install -q --upgrade "yt-dlp[default]" opencv-python-headless pandas==2.2.3


In [ ]:
from pathlib import Path
import hashlib,json,platform,re,shutil,subprocess,urllib.request,zipfile
import cv2,numpy as np,pandas as pd
from google.colab import files
from IPython.display import display

PINNED_COMMIT='2b4c5bafe72c51aa983805733d8c2635085cf905'
RAW=f'https://raw.githubusercontent.com/bariscelikk1/cALorie/{PINNED_COMMIT}/'
INPUTS={
 'ml/data/source_balance_v2/candidates.csv':'d69bdff1aed46349f67b695676debe6c767d11f3cda62844251084f9eca0aa17',
 'ml/data/source_balance_v2/supplemental_candidates.csv':'a550f6856fc50afa889ebe0501e2d5e335a67ceaf60bc241d9929c024b170221',
 'ml/data/source_balance_v2/dense_review_targets.csv':'846692d71451eade130c3e1014a070bea12dac507f16e3e40993fc849175abca',
 'ml/data/source_balance_v2/dense_visual_review.csv':'5888ada9b04573cdce82e94f72d72ff8d8647d79c52f53100e6222f5de3c910f',
 'ml/data/source_balance_v2/replacement_screening_review.csv':'0a2371dc5b439c23b410824b21b0797148737831ce32c1b3e1a7392a7428e061',
 'ml/data/youtube_creator_benchmark_v1/candidates.csv':'aa3063810fa3b0e9fe169c04281ef65cc33a442d18be0fa0e5fec7669e06268b',
 'ml/data/youtube_creator_benchmark_v1/excluded_source_inventory.csv':'ecf18c29ef5b7ba1eca3f45677e463df256b965888f3913ae55606807baf2232',
 'ml/data/youtube_mixed_workouts_v1/manifest.csv':'1d58ed25261a172e49abe3f9b7313236d339e5fbe067ecccf838c6f187efb3eb',
 'ml/data/youtube_long_workouts_v2/candidates.csv':'fb8af2597dfb93942bb2512e524352036ac6ea5e6ea2df37c1ddd29c0c78da34',
}
WORK=Path('/content/calorie_source_balance_v2_30c_work')
OUTPUT=Path('/content/calorie_source_balance_v2_replacement_dense_review')
for folder in (WORK,WORK/'videos',WORK/'frames',OUTPUT,OUTPUT/'dense_contact_sheets',OUTPUT/'metadata'):
 folder.mkdir(parents=True,exist_ok=True)

def sha256(path):
 d=hashlib.sha256()
 with Path(path).open('rb') as f:
  for block in iter(lambda:f.read(1024*1024),b''):d.update(block)
 return d.hexdigest()

def fetch(relative,digest):
 path=WORK/'pinned_inputs'/relative;path.parent.mkdir(parents=True,exist_ok=True)
 if not path.exists() or sha256(path)!=digest:
  tmp=path.with_suffix(path.suffix+'.part');tmp.unlink(missing_ok=True)
  urllib.request.urlretrieve(RAW+relative,tmp);assert sha256(tmp)==digest,relative;tmp.replace(path)
 return path

paths={name:fetch(name,digest) for name,digest in INPUTS.items()}
main=pd.read_csv(paths['ml/data/source_balance_v2/candidates.csv'])
supplemental=pd.read_csv(paths['ml/data/source_balance_v2/supplemental_candidates.csv'])
dense_targets=pd.read_csv(paths['ml/data/source_balance_v2/dense_review_targets.csv'])
dense_review=pd.read_csv(paths['ml/data/source_balance_v2/dense_visual_review.csv'])
replacement_review=pd.read_csv(paths['ml/data/source_balance_v2/replacement_screening_review.csv'])
reserved=pd.concat([pd.read_csv(paths[name]) for name in (
 'ml/data/youtube_creator_benchmark_v1/candidates.csv',
 'ml/data/youtube_creator_benchmark_v1/excluded_source_inventory.csv',
 'ml/data/youtube_mixed_workouts_v1/manifest.csv',
 'ml/data/youtube_long_workouts_v2/candidates.csv')],ignore_index=True)
normalize=lambda value:re.sub('[^a-z0-9]+','',str(value).casefold())
assert (~dense_review.training_allowed).all() and (~replacement_review.training_allowed).all()
advanced=replacement_review[replacement_review.screening_decision=='advance_dense_review'].copy()
assert len(advanced)==3 and advanced.candidate_intervals_s.notna().all()
rows=[]
for row in advanced.itertuples(index=False):
 start,end=map(float,str(row.candidate_intervals_s).split('-'))
 rows.append({'review_id':'replacement_'+row.video_id,'video_id':row.video_id,'target_class':row.target_class,'target_hypothesis':row.target_class,'start_s':start,'end_s':end})
max_target=dense_targets[dense_targets.review_id=='max_row_full_pass'].iloc[0]
assert dense_review.loc[dense_review.review_id=='max_row_full_pass','review_decision'].iloc[0]=='unavailable_retry_required'
rows.append({'review_id':'max_row_retry','video_id':max_target.video_id,'target_class':max_target.target_class,'target_hypothesis':max_target.target_hypothesis,'start_s':float(max_target.start_s),'end_s':float(max_target.end_s)})
targets=pd.DataFrame(rows)
all_intake=pd.concat([main,supplemental],ignore_index=True)
selected=all_intake[all_intake.video_id.isin(set(targets.video_id))].copy()
assert len(selected)==4 and selected.video_id.is_unique
assert set(selected.video_id).isdisjoint(set(reserved.video_id))
assert set(selected.creator.map(normalize)).isdisjoint(set(reserved.creator.map(normalize)))
assert (~selected.training_allowed).all()
print('CPU 30C:',len(targets),'dense evidence regions across',len(selected),'sources; training forbidden')
display(targets)


In [ ]:
def deno_args():
 asset={'x86_64':'deno-x86_64-unknown-linux-gnu.zip','aarch64':'deno-aarch64-unknown-linux-gnu.zip'}.get(platform.machine().lower())
 if not asset:return []
 try:
  deno=WORK/'deno'/'deno'
  if not deno.exists():
   deno.parent.mkdir(parents=True,exist_ok=True);archive=deno.parent/'deno.zip'
   urllib.request.urlretrieve('https://github.com/denoland/deno/releases/latest/download/'+asset,archive)
   with zipfile.ZipFile(archive) as z:z.extract('deno',deno.parent)
   deno.chmod(0o755)
  return ['--js-runtimes',f'deno:{deno}','--remote-components','ejs:github']
 except Exception as exc:print('Deno unavailable:',type(exc).__name__);return []
JS_ARGS=deno_args()

def run(args,timeout=300):
 try:
  p=subprocess.run(args,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,timeout=timeout,check=False);return p.returncode,p.stdout
 except subprocess.TimeoutExpired as exc:return 124,str(exc)

def info(path):
 code,out=run(['ffprobe','-v','error','-select_streams','v:0','-show_entries','stream=width,height,duration','-of','json',str(path)],30)
 if code:return None
 try:
  stream=json.loads(out)['streams'][0];return {'height':int(stream['height']),'duration_s':float(stream.get('duration') or 0)}
 except (ValueError,IndexError,KeyError):return None

def obtain(row):
 failures=[]
 formats=('18/best[height=360][ext=mp4]','best[height<=480][ext=mp4]/best[height<=480]','bv*[height<=480]+ba/b[height<=480]')
 clients=([],['--extractor-args','youtube:player_client=android_vr,web_creator'])
 for extra in clients:
  for fmt in formats:
   template=str(WORK/'videos'/(row.video_id+'.%(ext)s'))
   args=['yt-dlp','--no-playlist','--no-progress','--retries','3','--fragment-retries','3','--socket-timeout','25','-f',fmt,'--merge-output-format','mp4','-o',template,*extra,*JS_ARGS,row.source_url]
   code,out=run(args,360)
   matches=[p for p in (WORK/'videos').glob(row.video_id+'.*') if p.suffix in {'.mp4','.mkv','.webm'} and info(p)]
   if code==0 and matches:return matches[0],''
   failures.append(' '.join(out.splitlines()[-4:])[-500:])
 return None,' | '.join(failures[-3:])

source_paths={};source_status=[]
reserved_creators=set(reserved.creator.map(normalize))
reserved_channels=set(reserved.channel_id.dropna().astype(str)) if 'channel_id' in reserved else set()
for row in selected.itertuples(index=False):
 record={'video_id':row.video_id,'creator':row.creator,'status':'unavailable','error':''};print('Getting',row.video_id,row.creator)
 video,error=obtain(row)
 if video is None:record['error']=error;source_status.append(record);continue
 code,out=run(['yt-dlp','--skip-download','--dump-single-json','--no-warnings',*JS_ARGS,row.source_url],90)
 if code==0:
  raw=json.loads(out);metadata={key:raw.get(key) for key in ('id','title','channel','channel_id','duration','license','availability','webpage_url')}
  (OUTPUT/'metadata'/(row.video_id+'.json')).write_text(json.dumps(metadata,indent=2,ensure_ascii=False)+'\n')
  if metadata['id']!=row.video_id or metadata['channel_id']!=row.channel_id or normalize(metadata['channel']) in reserved_creators or metadata['channel_id'] in reserved_channels:
   record.update(status='identity_conflict',error='Observed source conflicts with pinned intake');source_status.append(record);continue
 source_paths[row.video_id]=video;record['status']='ready';source_status.append(record)
print('Video sources ready:',len(source_paths),'of',len(selected))


In [ ]:
def make_pages(frame_root,page_root,start_s):
 frames=sorted(frame_root.glob('*.jpg'));pages=0
 for offset in range(0,len(frames),12):
  tiles=[]
  for index in range(offset,min(offset+12,len(frames))):
   frame=cv2.imread(str(frames[index]));sec=float(start_s)+index/2;mm=int(sec//60);ss=sec-mm*60
   cv2.rectangle(frame,(0,0),(104,26),(0,0,0),-1);cv2.putText(frame,f'{mm:02d}:{ss:04.1f}',(5,18),cv2.FONT_HERSHEY_SIMPLEX,.5,(0,255,255),1);tiles.append(frame)
  while len(tiles)<12:tiles.append(np.zeros_like(tiles[0]))
  page=np.vstack([np.hstack(tiles[i:i+4]) for i in (0,4,8)]);cv2.imwrite(str(page_root/f'{pages:03d}.jpg'),page,[cv2.IMWRITE_JPEG_QUALITY,87]);pages+=1
 return pages,len(frames)

results=[]
for row in targets.itertuples(index=False):
 entry={'review_id':row.review_id,'video_id':row.video_id,'target_class':row.target_class,'start_s':row.start_s,'end_s':row.end_s,'status':'unavailable','frames':0,'pages':0,'error':''}
 video=source_paths.get(row.video_id)
 if video is not None and info(video) and row.end_s<=info(video)['duration_s']+1:
  frame_root=WORK/'frames'/row.review_id;page_root=OUTPUT/'dense_contact_sheets'/row.review_id
  frame_root.mkdir(parents=True,exist_ok=True);page_root.mkdir(parents=True,exist_ok=True)
  code,out=run(['ffmpeg','-hide_banner','-loglevel','error','-y','-ss',str(row.start_s),'-i',str(video),'-t',str(row.end_s-row.start_s),'-vf','fps=2,scale=320:180:force_original_aspect_ratio=decrease,pad=320:180:(ow-iw)/2:(oh-ih)/2:black','-q:v','3',str(frame_root/'%06d.jpg')],900)
  if code==0 and list(frame_root.glob('*.jpg')):
   pages,frames=make_pages(frame_root,page_root,row.start_s);entry.update(status='evidence_ready',pages=pages,frames=frames)
  else:entry.update(status='decode_failed',error=out[-500:])
 elif video is not None:entry.update(status='duration_mismatch',error=str(info(video)))
 else:entry['error']='Source not ready; see source_status.csv'
 results.append(entry);print(row.review_id,entry['status'],entry['pages'],'pages')

status=pd.DataFrame(results);status.to_csv(OUTPUT/'dense_review_status.csv',index=False)
pd.DataFrame(source_status).to_csv(OUTPUT/'source_status.csv',index=False)
targets.to_csv(OUTPUT/'dense_review_targets.csv',index=False)
for relative in ('ml/data/source_balance_v2/dense_visual_review.csv','ml/data/source_balance_v2/replacement_screening_review.csv'):
 shutil.copy2(paths[relative],OUTPUT/Path(relative).name)
metadata={'notebook':'30c_colab_source_balance_v2_replacement_dense_review.ipynb','runtime':'CPU','pinned_commit':PINNED_COMMIT,'dense_regions':len(targets),'evidence_ready':int((status.status=='evidence_ready').sum()),'training_allowed':False,'raw_video_exported':False,'landmarks_exported':False}
(OUTPUT/'run_metadata.json').write_text(json.dumps(metadata,indent=2)+'\n')
checksum_lines=[]
for path in sorted(p for p in OUTPUT.rglob('*') if p.is_file()):checksum_lines.append(f'{sha256(path)}  {path.relative_to(OUTPUT)}')
(OUTPUT/'SHA256SUMS.txt').write_text('\n'.join(checksum_lines)+'\n')
archive=shutil.make_archive(str(OUTPUT),'zip',OUTPUT)
display(status);print('Created:',archive,round(Path(archive).stat().st_size/1e6,1),'MB');files.download(archive)


The returned sheets are **review evidence only**. They do not approve labels, source rights, person independence, pose quality, or training use. Inspect exact motion boundaries next; only then may approved intervals proceed to a separate pose audit and independent human review.